# 2.2.1 智能信用评分Logistic回归模型开发与测试

## 任务说明
互联网金融飞速发展，使得个人金融理财变得越来越容易。而其中信用评分技术是一种对贷款申请人（信用卡申请人）做风险评估分值的统计模型，可以根据客户提供的资料、客户的历史数据、第三方平台数据（芝麻分、京东、微信等），对客户的信用进行评估。

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        导入必要的库                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 📦 pandas：Python数据分析核心库
# ───────────────────────────────────────────────────────────────────────────────
# 【通俗理解】就像一个超级强大的Excel表格处理工具，可以轻松读取、处理、分析各种格式的数据文件
# 【主要用途】读取CSV/Excel文件、数据清洗、数据筛选、数据统计等
# 【常用方法】
#   - pd.read_csv()：读取CSV文件
#   - df.head()：查看前几行数据
#   - df.drop()：删除列或行
#   - df.describe()：查看数据统计信息
# 【专业术语】DataFrame：pandas的核心数据结构，类似二维表格
import pandas as pd

# ───────────────────────────────────────────────────────────────────────────────
# 🔪 train_test_split：数据集划分工具
# ───────────────────────────────────────────────────────────────────────────────
# 【通俗理解】把数据切成两份，一份用来"学习"（训练集），一份用来"考试"（测试集）
# 【为什么需要】就像学生学习和考试一样，不能用同一套题学习和考试，否则无法检验真实水平
# 【参数说明】
#   - test_size=0.2：测试集占20%，训练集占80%
#   - random_state=42：随机种子，保证每次运行结果一致（42是程序员的传统幸运数字）
# 【专业术语】
#   - 训练集(Training Set)：用于训练模型的数据
#   - 测试集(Test Set)：用于评估模型性能的数据
#   - 过拟合(Overfitting)：模型在训练集表现好，测试集表现差，相当于"死记硬背"
from sklearn.model_selection import train_test_split



# ───────────────────────────────────────────────────────────────────────────────
# 🥒 pickle：Python对象序列化工具
# ───────────────────────────────────────────────────────────────────────────────
# 【通俗理解】把Python对象"打包"保存到文件，之后可以"拆包"恢复使用
# 【为什么需要】训练好的模型需要保存，否则每次都要重新训练，太浪费时间
# 【类比】就像把做好的菜冷冻保存，下次想吃直接加热，不用重新做
# 【常用方法】
#   - pickle.dump(obj, file)：将对象保存到文件
#   - pickle.load(file)：从文件加载对象
# 【专业术语】
#   - 序列化(Serialization)：将对象转换为可存储的格式
#   - 反序列化(Deserialization)：从存储格式恢复对象
#   - .pkl文件：pickle格式的文件扩展名
import pickle



# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        加载数据                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# 【pd.read_csv()参数说明】
#   - 第一个参数：文件路径
#   - encoding：文件编码（中文文件名可能需要指定）
# 【返回值】DataFrame对象，类似Excel表格
data = pd.read_csv('data/finance数据集.csv')

# 【data.head(n)】显示前n行数据，默认n=5
# 【其他常用方法】
#   - data.tail(n)：显示后n行
#   - data.shape：显示数据维度（行数, 列数）
#   - data.info()：显示数据类型和缺失值信息
#   - data.describe()：显示数值型字段的统计信息
print(data.head())

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     📊 数据表字段解读                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ┌──────────────────────────────────────────────────────────────────────────────┐
# │                        各列含义详解                                           │
# ├──────────────────────────────────────────────────────────────────────────────┤
# │ 列名                                    │ 中文含义      │ 说明               │
# ├─────────────────────────────────────────┼───────────────┼────────────────────┤
# │ Unnamed: 0                              │ 序号          │ 无意义，分析时删除  │
# │ SeriousDlqin2yrs                        │ ⭐两年内违约  │ 1=违约, 0=未违约   │
# │ RevolvingUtilizationOfUnsecuredLines    │ 信用额度使用率│ 越高风险越大        │
# │ age                                     │ 年龄          │ 客户年龄           │
# │ NumberOfTime30-59DaysPastDueNotWorse    │ 30-59天逾期   │ 逾期次数           │
# │ DebtRatio                               │ 负债比率      │ 越高风险越大        │
# │ MonthlyIncome                           │ 月收入        │ 客户月收入         │
# │ NumberOfOpenCreditLinesAndLoans         │ 信贷账户数    │ 开放的信贷账户数量  │
# │ NumberOfTimes90DaysLate                 │ 90天以上逾期  │ 严重逾期记录        │
# │ NumberRealEstateLoansOrLines            │ 房贷数量      │ 房地产贷款数量      │
# │ NumberOfTime60-89DaysPastDueNotWorse    │ 60-89天逾期   │ 逾期次数           │
# │ NumberOfDependents                      │ 家属数量      │ 受抚养人数         │
# └──────────────────────────────────────────────────────────────────────────────┘

# 【关键理解】
# - 目标变量：SeriousDlqin2yrs - 模型要预测的就是这个值（是否违约）
# - 特征变量：其他所有列都是用来预测的依据
# - 风险指标：信用额度使用率、负债率、逾期次数越高，违约风险越大

# 【案例分析 - 第0行客户（高风险）】
# - 违约了（SeriousDlqin2yrs=1）
# - 信用额度使用率76.6%，负债率80.3%，都很高
# - 有2次30-59天逾期记录
# - 月收入9120元，但有13个信贷账户


   Unnamed: 0  SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines  age  \
0           1                 1                              0.766127   45   
1           2                 0                              0.957151   43   
2           3                 0                              0.640255   58   
3           4                 0                              0.222222   35   
4           5                 1                              0.888889   62   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
0                                     2   0.802982         9120.0   
1                                     1   0.640255         2678.0   
2                                     0   0.937641         4000.0   
3                                     0   0.250000         3500.0   
4                                     3   0.750000         2200.0   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
0                               13                   

In [41]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     数据预处理与划分                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 🎯 选择自变量(特征)和因变量(标签)
# ───────────────────────────────────────────────────────────────────────────────
# 【专业术语】
#   - 自变量/特征(Features/X)：用来预测的依据，如年龄、收入等
#   - 因变量/标签(Label/y)：要预测的结果，如"是否违约"
# 【代码解释】
#   - data.drop(['SeriousDlqin2yrs', 'Unnamed: 0'], axis=1)：
#     删除'SeriousDlqin2yrs'（目标变量）和'Unnamed: 0'（无意义的序号列）
#   - axis=1：表示按列删除（axis=0是按行删除）

# X：特征矩阵（自变量）- 包含所有用来预测的信息
# X = data.drop(['SeriousDlqin2yrs', 'Unnamed: 0'], axis=1)
X = data.drop(['Unnamed: 0'], axis=1)
# y：目标变量（因变量）- 我们要预测的结果
# SeriousDlqin2yrs：两年内是否发生严重违约（0=否，1=是）
y = data['SeriousDlqin2yrs']

# ───────────────────────────────────────────────────────────────────────────────
# ✂️ 分割训练集和测试集
# ───────────────────────────────────────────────────────────────────────────────
# 【train_test_split参数说明】
#   - X, y：要分割的特征和标签
#   - test_size=0.2：测试集占20%，训练集占80%
#   - random_state=42：随机种子，保证每次运行结果一致
#     （为什么是42？这是程序员的传统幸运数字，来自《银河系漫游指南》）
# 【返回值】四个变量：
#   - X_train：训练集特征
#   - X_test：测试集特征
#   - y_train：训练集标签
#   - y_test：测试集标签
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     训练Logistic回归模型                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# ───────────────────────────────────────────────────────────────────────────────
# 🤖 LogisticRegression：逻辑回归分类器
# ───────────────────────────────────────────────────────────────────────────────
# 【通俗理解】一种用来做"二选一"预测的算法，比如预测"会不会违约"、"是不是垃圾邮件"
# 【为什么叫逻辑回归】虽然名字里有"回归"，但它其实是分类算法！
#   因为它用逻辑函数(Sigmoid函数)把结果压缩到0-1之间，表示概率
# 【工作原理】想象一条曲线，把数据分成两类：曲线一边是"好客户"，另一边是"坏客户"
# 【应用场景】
#   - 信用评分(违约/不违约)
#   - 医疗诊断(患病/健康)
#   - 垃圾邮件识别(垃圾/正常)
# 【参数说明】
#   - max_iter=1000：最大迭代次数，就像"最多学习1000遍"
# 【专业术语】
#   - 分类器(Classifier)：一种能将数据分到不同类别的算法模型
#   - Sigmoid函数：将任意数值转换为0-1之间的概率值
#   - 决策边界：区分不同类别的分界线
from sklearn.linear_model import LogisticRegression
lr = dir(LogisticRegression)
print([x for x in lr if not x.startswith('_')])
print(len(lr))
# ───────────────────────────────────────────────────────────────────────────────
# 🤖 创建逻辑回归模型
# ───────────────────────────────────────────────────────────────────────────────
# 【LogisticRegression参数说明】
#   - max_iter=1000：最大迭代次数
#     【通俗理解】模型"学习"的最大遍数，就像学生最多复习1000遍
#     【为什么需要】默认值可能不够，导致模型没学完就停止了
#   - 其他常用参数：
#     - C=1.0：正则化强度的倒数，越小正则化越强（防止过拟合）
#     - penalty='l2'：正则化类型，l1或l2
#     - solver='lbfgs'：优化算法
model = LogisticRegression(max_iter=1000)
model = LogisticRegression(max_iter=1000, random_state=42)

# ───────────────────────────────────────────────────────────────────────────────
# 🎓 训练模型（拟合）
# ───────────────────────────────────────────────────────────────────────────────
# 【model.fit(X_train, y_train)】
# 【通俗理解】让模型"学习"训练数据，找出特征与标签之间的关系
# 【类比】就像学生通过课本（训练集）学习知识
# 【专业术语】
#   - 拟合(Fitting)：让模型学习数据中的规律
#   - 参数(Parameters)：模型学习到的内部变量（权重和偏置）
model.fit(X_train, y_train)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [43]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     保存模型与预测                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 💾 保存模型到文件
# ───────────────────────────────────────────────────────────────────────────────
# 【pickle.dump()参数说明】
#   - 第一个参数：要保存的对象（这里是训练好的模型）
#   - 第二个参数：文件对象（'wb'表示以二进制写入模式打开）
# 【文件扩展名】.pkl 是pickle的标准扩展名
# 【为什么要保存】训练模型可能需要很长时间，保存后下次可以直接加载使用
with open('2.2.1_model.pkl', 'wb') as file:
    pickle.dump(model, file)
with open('2.2.1_model.pkl', 'rb') as file:
    pickle.dump(modle, file)    
# ───────────────────────────────────────────────────────────────────────────────
# 🔮 使用模型进行预测
# ───────────────────────────────────────────────────────────────────────────────
# 【model.predict(X_test)】
# 【通俗理解】用训练好的模型对测试集进行预测，就像学生参加考试
# 【返回值】预测的类别标签（0或1）
# 【其他预测方法】
#   - model.predict_proba(X_test)：返回预测概率（每个类别的概率）
#   - model.predict_log_proba(X_test)：返回对数概率
y_pred = model.predict(X_test)
y_pred = model.predict(X_test)

# ───────────────────────────────────────────────────────────────────────────────
# 📝 保存预测结果
# ───────────────────────────────────────────────────────────────────────────────
# 【pd.DataFrame()】创建数据框
# 【to_csv()】保存为CSV文件
#   - index=False：不保存行索引
pd.DataFrame(y_pred, columns=['预测结果']).to_csv('2.2.1_results.txt', index=False)
pd.DataFrame(y_pred, columns['预测结果']).to_csv('2.2.1_results.txt', index=False)
# ───────────────────────────────────────────────────────────────────────────────
# 📊 生成分类评估报告
# ───────────────────────────────────────────────────────────────────────────────
# 【classification_report参数说明】
#   - y_test：真实标签
#   - y_pred：预测标签
#   - zero_division=1：当某个类别没有预测样本时，返回1而不是报错
# ───────────────────────────────────────────────────────────────────────────────
# 📊 classification_report：分类模型评估报告生成器
# ───────────────────────────────────────────────────────────────────────────────
# 【通俗理解】给模型打一份"成绩单"，告诉你模型表现如何
# 【包含指标】精确率(Precision)、召回率(Recall)、F1分数(F1-Score)、准确率(Accuracy)
# 【专业术语解释】
#   - 精确率(Precision)：预测为正例中真正是正例的比例
#     例子：预测违约的人中，真正违约的比例
#     公式：TP / (TP + FP)
#     通俗说："我说你有问题，你真有问题的概率"
#   - 召回率(Recall)：真正的正例中被正确预测的比例
#     例子：真正违约的人中，被正确识别的比例
#     公式：TP / (TP + FN)
#     通俗说："你有问题，我能发现你的概率"
#   - F1分数(F1-Score)：精确率和召回率的调和平均，综合评价指标
#     公式：2 * (Precision * Recall) / (Precision + Recall)
#     通俗说："综合考虑抓得准不准和抓得全不全"
#   - 准确率(Accuracy)：所有预测中预测正确的比例
#     公式：(TP + TN) / (TP + TN + FP + FN)
#     通俗说："整体预测对了多少"
#   其中：TP=真阳性, TN=真阴性, FP=假阳性, FN=假阴性
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred, zero_division=1)
with open('2.2.1_report.txt', 'w') as file:
    file.write(report)

report = classification_report(y_test, y_pred, zero_division=1)
with open('2.2.1_report.txt', 'w') as file:
    file.write(report)

NameError: name 'modle' is not defined

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     分析测试结果                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 📈 计算模型准确率
# ───────────────────────────────────────────────────────────────────────────────
# 【准确率计算】(y_test == y_pred).mean()
# 【通俗理解】预测正确的比例
# 【分解说明】
#   - y_test == y_pred：比较真实值和预测值，返回布尔数组
#   - .mean()：计算True的比例（True=1, False=0的平均值）
# 【注意】准确率高不一定代表模型好！
#   如果数据不平衡（如违约率只有6%），模型把所有人都预测为不违约，
#   准确率也能达到94%，但这样的模型毫无价值！
accuracy = (y_test == y_pred).mean()
print(f"模型准确率: {accuracy:.2f}")

accuracy = (y_test == y_pred).mean()
print(f"accuracy: {accuracy:.2f}")

模型准确率: 0.93


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                处理数据不平衡（SMOTE过采样）                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# ───────────────────────────────────────────────────────────────────────────────
# ⚖️ SMOTE：合成少数类过采样技术 (Synthetic Minority Oversampling Technique)
# ───────────────────────────────────────────────────────────────────────────────
# 【通俗理解】当数据"一边倒"时，人为制造一些少数类的数据来平衡
# 【为什么需要】比如1000个客户中只有50个违约，模型会"偷懒"把所有人都预测为不违约
#   这样准确率也有95%，但实际上完全没有识别出违约客户！
# 【解决方法】SMOTE通过"插值"方法，在少数类样本之间生成新的样本，让数据更平衡
# 【工作原理】
#   1. 找到少数类样本的最近邻
#   2. 在样本与最近邻之间的连线上随机生成新样本
# 【专业术语】
#   - 数据不平衡(Imbalanced Data)：各类别样本数量差异悬殊
#   - 过采样(Oversampling)：增加少数类样本数量
#   - 欠采样(Undersampling)：减少多数类样本数量
#   - 插值(Interpolation)：在两个已知值之间估算新值
from imblearn.over_sampling import SMOTE

# ───────────────────────────────────────────────────────────────────────────────
# ⚖️ 使用SMOTE处理数据不平衡, 创建平衡数据集
# ───────────────────────────────────────────────────────────────────────────────
# 【SMOTE工作原理】
# 1. 对于每个少数类样本，找到它的k个最近邻
# 2. 在样本与最近邻之间的连线上随机选择一点作为新样本
# 3. 重复上述过程，直到少数类样本数量与多数类接近
# 【参数说明】
#   - random_state=42：随机种子，保证结果可复现
# 【fit_resample()】拟合并重采样，返回平衡后的数据
smote = SMOTE(random_state=42)
smote = SMOTE(random_state=42)
#print(dir(smote))
X_resampled, y_resampled = smote.fit_resample(x_train, y_train)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# ───────────────────────────────────────────────────────────────────────────────
# 🔄 使用平衡后的数据重新训练模型
# ───────────────────────────────────────────────────────────────────────────────
model.fit(X_resampled, y_resampled)
model.fit(X_resampled, y_resampled)

# ───────────────────────────────────────────────────────────────────────────────
# 🔮 重新预测
# ───────────────────────────────────────────────────────────────────────────────
y_pred_resampled = model.predict(X_test)
y_pred_resampled = modle.predict(X_test)

# ───────────────────────────────────────────────────────────────────────────────
# 📝 保存新预测结果
# ───────────────────────────────────────────────────────────────────────────────
pd.DataFrame(y_pred_resampled, columns=['预测结果']).to_csv('2.2.1_results_xg.txt', index=False)
#pd.DataFrame(y_pred_resampled, columns=['yucejieguo']).to_csv('xg.txt', index=False)

# ───────────────────────────────────────────────────────────────────────────────
# 📊 生成新的评估报告
# ───────────────────────────────────────────────────────────────────────────────
report_resampled = classification_report(y_test, y_pred_resampled, zero_division=1)
report_resampled = classification_report(y_test, y_pred_resampled, zero_division=1)
with open('xg.txt', 'w') as file:
    file.write(report_resampled)
with open('2.2.1_report_xg.txt', 'w') as file:
    file.write(report_resampled)

['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_build_request_for_signature', '_check_X_y', '_check_feature_names', '_check_n_features', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_estimator_type', '_fit_resample', '_generate_samples', '_get_default_requests', '_get_doc_link', '_get_metadata_request', '_get_param_names', '_get_tags', '_in_danger_noise', '_make_samples', '_more_tags', '_parameter_constraints', '_repr_html_', '_repr_html_inner', '_repr_mimebundle_', '_sampling_strategy_docstring', '_sampling_type', '_validate_data', '_v

/Users/guoruidong/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     分析SMOTE处理后的结果                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 📊 计算SMOTE处理后的准确率
# ───────────────────────────────────────────────────────────────────────────────
# 【注意】SMOTE处理后，准确率可能会下降，但召回率通常会提升
# 这是正常的权衡：我们牺牲一些准确率，换取更好地识别少数类（违约客户）
# 在信用评分场景中，漏掉一个违约客户的代价往往比误判一个好客户更大
accuracy_resampled = (y_test == y_pred_resampled).mean()
accuracy_resampled = (y_test == y_pred_resampled).mean()
print(f"重新采样后的模型准确率: {accuracy_resampled}, old: {accuracy}")

重新采样后的模型准确率: 0.9333333333333333, old: 0.9333333333333333
